# Sanity and information checks: monthly firm variables

Checks every imported time-series firm variable against the filtered U.S. price universe and against each
other. All functions live in `src/datastream/firm_evaluation.py`; this notebook only configures and displays.
Tables are written to `OUTPUT_DIR` as CSV so results can be compared across downloads.

| Block | Question | Main outputs |
|---|---|---|
| **A Coverage** | For which firms and years is the variable there? | EW/VW coverage of the price universe, coverage by size group, firm-years, history gaps, matching |
| **B Timing** | When and how often does the value change, and does it stop when the stock dies? | changes per firm-year, reporting lag after fiscal year end, stale runs, values after the last price |
| **C Values** | Are the values plausible? | yearly quantiles and means, zeros/negatives, robust-z outliers, unit jumps (x1000 errors) |
| **D Consistency** | Do variables agree with each other and with market data? | identities and bounds from `config/firm_relations.csv`, recomputed ratios, alignment of update months |

**Prerequisites:** variables imported with `scripts/03_import_firm_variables.py`, the monthly universe built
with `scripts/05_build_monthly_universe.py`, and (for the reporting lag) the static WC05350 imported.

The summary flags use the heuristic thresholds in `firm_evaluation.THRESHOLDS`; they mark what to look at,
not what is wrong. Values are checked as downloaded (monthly Datastream series, dates moved to month end).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from datastream import firm_evaluation as fe
from datastream.preprocessing.firm_data import VARIABLES_SUBDIR, load_variable, read_registry
from datastream.preprocessing.static_data import load_static, static_panel_path

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
fe.set_style()

# ── configuration ───────────────────────────────────────────────────────────────────────────────────
REGION = "US"
FIRM_ROOT = Path(os.environ.get("DS_FIRM_ROOT", "D:/Datastream/Firmcharacteristics_Monthly")) / REGION
PRICE_PATH = Path(os.environ.get("DS_PRICE_PATH", "D:/Datastream/PriceData/US/processed"))
PENNY_PERCENTILE = "0.2"
UNIVERSE_FILE = PRICE_PATH / f"monthly_universe_{PENNY_PERCENTILE}.parquet"
VARIABLES = None                   # None = all imported variable panels, or e.g. ["WC02999", "WC01001"]
FYE_VARIABLE = "WC05350"           # static fiscal-year end (for the reporting lag)
ALIGNMENT_CATEGORIES = ["balance_sheet", "income_statement", "cash_flow"]   # for D2
OUTPUT_DIR = FIRM_ROOT / "Paneldata" / "checks"

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
REGISTRY_FILE = REPO / "config" / "firm_variables.csv"
RELATIONS_FILE = REPO / "config" / "firm_relations.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and run the checks

In [ ]:
uni = pd.read_parquet(UNIVERSE_FILE)
print(f"Universe: {len(uni):,} stock-months, {uni['DSCD'].nunique():,} stocks, "
      f"{uni['Date'].min():%Y-%m} to {uni['Date'].max():%Y-%m}")

registry = read_registry(REGISTRY_FILE)
signs = dict(zip(registry["variable"], registry.get("sign", pd.Series("", index=registry.index))))
categories = dict(zip(registry["variable"], registry.get("category", pd.Series("", index=registry.index))))

available = sorted(p.stem for p in (FIRM_ROOT / VARIABLES_SUBDIR).glob("*.parquet"))
variables = [v for v in (VARIABLES or available) if v in available]
print(f"{len(variables)} variables: {variables}")

fye = None
if static_panel_path(FIRM_ROOT, FYE_VARIABLE).exists():
    fye = fe.fye_months(load_static(FIRM_ROOT, FYE_VARIABLE), FYE_VARIABLE)
    print(f"Fiscal-year end available for {len(fye):,} firms")
else:
    print(f"{FYE_VARIABLE} not imported: reporting lag is skipped")

results = {}
for v in variables:
    results[v] = fe.check_variable(load_variable(FIRM_ROOT, v), v, uni, fye, signs.get(v, ""))
    print(f"  {v}: done")

## 2. Summary

One row per variable. `flags` lists the checks outside the thresholds below; the sections after this one
show the details behind each number.

In [ ]:
summary = fe.summary_table([r["summary"] for r in results.values()])
summary.insert(0, "category", [categories.get(v, "") for v in summary.index])
print("Thresholds:", fe.THRESHOLDS)
display(summary[["category", "n_flags", "flags", "first_date", "n_firms", "first_year_vw_cov_50",
                 "ew_coverage_last_year", "vw_coverage_last_year", "median_changes_per_year",
                 "modal_lag_months", "share_dead_firms_padded", "share_stale_while_trading",
                 "share_sign_violations", "share_extreme", "n_unit_jumps"]])

## A. Coverage
### A1. Coverage of the price universe over time
EW: share of the stocks in the filtered price universe that have a value in that month. VW: their share of
total market cap. A large gap between the two lines means small stocks are missing.

In [ ]:
fig = fe.plot_coverage(results)
fig.savefig(OUTPUT_DIR / "A1_coverage.png", dpi=130, bbox_inches="tight")
plt.show()

### A2. Coverage by size group
Average monthly EW coverage per year and size group (Q1 = smallest; NYSE breakpoints when the statics contain
exchange codes). Shows in which years a variable is effectively a large-cap variable.

In [ ]:
fig = fe.plot_coverage_by_size(results)
fig.savefig(OUTPUT_DIR / "A2_coverage_by_size.png", dpi=130, bbox_inches="tight")
plt.show()

### A3. Firm-years per year and history, matching with the price universe

In [ ]:
firms_per_year = pd.DataFrame({v: r["yearly"]["n_firms"] for v, r in results.items()}).astype("Int64")
display(firms_per_year.T)

cols = ["n_firms", "median_history_months", "share_firms_with_gaps", "share_gap_months",
        "share_universe_firms_ever_covered", "share_variable_firms_in_universe", "outside_universe_share"]
display(summary[cols])

### A4. Levels over time
Median and 1%-trimmed mean of the firm-year values (last value of each firm in each calendar year). A step
in the median that is not visible in the economy points to a change in units, definitions or coverage.

In [ ]:
fig = fe.plot_yearly_levels(results)
plt.show()

## B. Timing
### B1. Value changes per firm-year
Firm-years with at least 12 monthly values. Annual items should change about once a year, interim items up to
four times; many firm-years with 0 changes mean stale data.

In [ ]:
fig = fe.plot_update_frequency(results)
fig.savefig(OUTPUT_DIR / "B1_update_frequency.png", dpi=130, bbox_inches="tight")
plt.show()

### B2. Reporting lag after fiscal year end
Month of each value change relative to the fiscal-year-end month (current WC05350). The mode is the typical
delay between fiscal year end and availability in Datastream, which matters for point-in-time merges.

In [ ]:
lag = pd.DataFrame({v: r["lag"]["share"] for v, r in results.items() if len(r["lag"])}).T
if len(lag):
    display(lag.reindex(columns=range(12)).fillna(0).style.format("{:.2f}").background_gradient(axis=1, cmap="Blues"))

### B3. Stale values and values after the last price
* `share_after_last_price`: observations after the stock's last month in the filtered price universe.
* `share_dead_firms_padded`: delisted stocks with values more than 12 months after their last price.
* `share_stale_while_trading`: observations in a run of identical values longer than `STALE_MONTHS` while the
  stock is still in the universe. Ratios based on prices (PE, DY) should rarely be stale.

In [ ]:
display(summary[["share_after_last_price", "share_after_delisting", "share_dead_firms_padded",
                 "share_before_first_price", "share_stale", "share_stale_while_trading",
                 "share_firms_with_stale_run"]])

## C. Values
### C1. Cross-sectional distribution per year
Bands: p5-p95 and p25-p75 of firm-year values, line: median. Log scale for non-negative variables.

In [ ]:
fig = fe.plot_quantiles(results, signs)
fig.savefig(OUTPUT_DIR / "C1_quantiles.png", dpi=130, bbox_inches="tight")
plt.show()

### C2. Zeros, negatives and extreme values
Sign violations are counted for variables marked `nonneg` in the registry. Extreme: |robust z| > `EXTREME_Z`
within the year (median/MAD, on log10 for non-negative variables and on asinh(x / median |x|) otherwise).

In [ ]:
display(summary[["share_zero", "share_negative", "share_sign_violations", "share_extreme"]])
extremes = pd.concat({v: r["extremes"] for v, r in results.items() if len(r["extremes"])},
                     names=["variable"]).reset_index(level=0)
extremes.to_csv(OUTPUT_DIR / "C2_extreme_values.csv", index=False)
display(extremes.sort_values("robust_z", key=abs, ascending=False).head(20))

### C3. Unit jumps
A single change by a factor of at least ~316 (`jump`), or a change by at least ~8 that is undone within two
years (`reversal`, e.g. one year reported in the wrong unit). One row per episode.

In [ ]:
jump_tables = {v: r["jumps"] for v, r in results.items() if len(r["jumps"])}
if jump_tables:
    jumps = pd.concat(jump_tables, names=["variable"]).reset_index(level=0).reset_index(drop=True)
    jumps.to_csv(OUTPUT_DIR / "C3_unit_jumps.csv", index=False)
    display(jumps.groupby(["variable", "type"]).size().unstack(fill_value=0))
    display(jumps.head(20))
else:
    print("No unit jumps found.")

## D. Consistency across variables
### D1. Identities, bounds and recomputed ratios
Evaluated on December cross-sections (one row per firm and year). `compare` rows are informational: the
definitions differ (e.g. average vs. year-end equity), so look at the median ratio and rank correlation.
The `mtbv` row also checks units: MV is in millions, Worldscope items are assumed to be in thousands.
Relations with variables that are not imported yet are skipped.

In [ ]:
relations = fe.read_relations(RELATIONS_FILE)
needed = sorted(set().union(*[fe._names(e) for e in relations["lhs"].tolist() + relations["rhs"].tolist()]))
dec = fe.december_panel(FIRM_ROOT, [v for v in needed if v in available], uni)
rel_table, rel_by_year, rel_examples = fe.evaluate_relations(dec, relations)
rel_table.to_csv(OUTPUT_DIR / "D1_relations.csv", index=False)
display(rel_table[["relation", "kind", "status", "n_firm_years", "share_violations", "median_ratio",
                   "share_ratio_within_5pct", "spearman", "note"]])

In [ ]:
if len(rel_by_year.columns):
    fig = fe.plot_relation_violations(rel_by_year)
    fig.savefig(OUTPUT_DIR / "D1_violations_by_year.png", dpi=130, bbox_inches="tight")
    plt.show()
for name, ex in rel_examples.items():
    ex.to_csv(OUTPUT_DIR / f"D1_examples_{name}.csv", index=False)

### D2. Alignment of update months
For each pair of accounting variables: share of firm-years (both with exactly one change) in which the value
changes in the same month. Items from the same annual report should be close to 1; low values mean the
series are stamped differently, which matters when variables are combined into ratios.

In [ ]:
align_vars = [v for v in variables if categories.get(v, "") in ALIGNMENT_CATEGORIES]
if len(align_vars) >= 2:
    alignment = fe.update_alignment({v: results[v]["panel"] for v in align_vars})
    alignment.to_csv(OUTPUT_DIR / "D2_update_alignment.csv")
    fig = fe.plot_matrix(alignment, "Share of firm-years with the update in the same month")
    fig.savefig(OUTPUT_DIR / "D2_update_alignment.png", dpi=130, bbox_inches="tight")
    plt.show()

## 3. Save tables

In [ ]:
summary.to_csv(OUTPUT_DIR / "summary.csv")
firms_per_year.to_csv(OUTPUT_DIR / "A3_firms_per_year.csv")
pd.concat({v: r["coverage"] for v, r in results.items()}, names=["variable"]).to_csv(OUTPUT_DIR / "A1_coverage.csv")
pd.concat({v: r["coverage_by_size"] for v, r in results.items() if len(r["coverage_by_size"])},
          names=["variable"]).to_csv(OUTPUT_DIR / "A2_coverage_by_size.csv")
pd.concat({v: r["yearly"] for v, r in results.items()}, names=["variable"]).to_csv(OUTPUT_DIR / "A4_yearly.csv")
pd.concat({v: r["updates"] for v, r in results.items()}, names=["variable"]).to_csv(OUTPUT_DIR / "B1_updates.csv")
if len(lag):
    lag.to_csv(OUTPUT_DIR / "B2_reporting_lag.csv")
pd.concat({v: r["quantiles"] for v, r in results.items()}, names=["variable"]).to_csv(OUTPUT_DIR / "C1_quantiles.csv")
print(f"Tables and figures written to {OUTPUT_DIR}")